## 1. ECG 切片

In [4]:
import os, re, shutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk

data_path = './data_parquets/'
file_name = data_path + 'ecg_log_2026-03-01_004716.parquet'
ecg_time_str = re.search(r'\d{4}-\d{2}-\d{2}_\d{6}', file_name).group()
data = pd.read_parquet(file_name)
window_size = 10 * 250
chunks = [data[i:i + window_size] for i in range(0, len(data), window_size)][:-1]

## 2. 数据百分比分布

In [5]:
from ecg_core.rr_hr_hrv import *

def nan_in_rrs(chunk):
    my_beats = BeatCalc(chunk)
    my_rr_s, my_rr_clean = my_beats.get_clean_rr()
    return my_rr_s.hasnans

def describe_percentiles(x):
    percentiles = [0.001, 0.01, 0.05, 0.10,
                   0.25, 0.50, 0.75,
                   0.90, 0.99, 0.999]

    qs = x.quantile(percentiles)

    qs.index = [f"{int(p*1000)/10:.1f}%" if p < 0.01
                else f"{int(p*100)}%" 
                for p in percentiles]

    return pd.concat([
        pd.Series({'min': x.min()}),
        qs,
        pd.Series({'max': x.max()})
    ])

data[data['lead'] == 0].apply(describe_percentiles)
#describe_percentiles(data['ecg'])    

,time,ecg,lead
min,1.772297e+09,0.0,0.0
0.1%,1.772297e+09,111.0,0.0
1%,1.772298e+09,300.0,0.0
5%,1.772299e+09,315.0,0.0
10%,1.772300e+09,318.0,0.0
25%,1.772305e+09,322.0,0.0
50%,1.772313e+09,327.0,0.0
75%,1.772321e+09,338.0,0.0
90%,1.772325e+09,357.0,0.0
99%,1.772328e+09,491.0,0.0


## 3. 脚本划分 good 和 bad

In [6]:
labels = np.zeros(len(chunks))
for i, chunk in enumerate(chunks):
    if 1 in chunk['lead']:
        labels[i] = 1
    elif nan_in_rrs(chunk):
        labels[i] = 1
    elif len(chunk[chunk['ecg']>600]) > 10 or len(chunk[chunk['ecg']<100])>30:
        labels[i] = 1

good_idx = np.where(labels == 0)[0]
bad_idx = np.where(labels == 1)[0]

good = [chunks[i] for i in good_idx]
bad = [chunks[i] for i in bad_idx]

print(len(good))
print(len(bad))

2868
229


## 4. 画出所有 Bad 样本

In [7]:
def save_chunk_plot(chunk, idx, label, base_dir=f"./ecg_figs/{ecg_time_str}", fs=250):
    # 建目录：ecg_time_str/good, ecg_time_str/bad, ecg_time_str/bad2good
    save_dir = os.path.join(base_dir, label)
    os.makedirs(save_dir, exist_ok=True)

    ecg = chunk["ecg"].copy()
    ecg.index = np.arange(len(ecg))
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 4))

    ax.plot(ecg, alpha=0.4, label="Raw")
    my_r_peaks = BeatCalc(chunk).peaks
    ax.scatter(my_r_peaks,
               ecg.loc[my_r_peaks],
               color="C0", s=20)
    
    try:
        ecg_signals, _ = nk.ecg_process(ecg, sampling_rate=fs)
        ax.plot(ecg_signals["ECG_Clean"], label="Clean")
        nk_r_peaks = ecg_signals.index[ecg_signals["ECG_R_Peaks"] != 0]
        ax.scatter(nk_r_peaks,
                ecg_signals.loc[nk_r_peaks, "ECG_Clean"],
                color="r", s=20)
    except Exception as e:
        print(f"警告：Chunk {i} 烂到无法处理，仅绘制原始波形。错误原因: {e}")
        # 仅绘制 Raw 数据，不依赖 NK2 的处理结果


    ax.set_title(f"Chunk {idx} | {label}")
    ax.set_xlabel("Samples")
    ax.set_ylabel("Amplitude")
    ax.legend(loc="upper right")

    filename = f"{idx}_chunk_{label}.jpg"
    path = os.path.join(save_dir, filename)

    plt.tight_layout()
    plt.savefig(path, dpi=200)
    plt.close(fig)

label_map = {0: "good", 1: "bad", 2: "bad2good"}

for i, chunk in enumerate(chunks):
    label_name = label_map[labels[i]]
    if label_name == 'bad':
        save_chunk_plot(chunk, i, label_name)

d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 2089 烂到无法处理，仅绘制原始波形。错误原因: integer division or modulo by zero


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 2091 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


...

警告：Chunk 2108 烂到无法处理，仅绘制原始波形。错误原因: integer division or modulo by zero


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 2109 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer


d:\Softwares\Miniconda\envs\ecg_env\lib\site-packages\neurokit2\signal\signal_period.py:84: NeuroKitWarning: Too few peaks detected to compute the rate. Returning empty vector.
  warn(


警告：Chunk 2110 烂到无法处理，仅绘制原始波形。错误原因: cannot convert float NaN to integer


## 5. 筛选bad2good

In [35]:
bad2good_idxs = {
    '2026-02-26_235435': [
        0, 1, 55, 69, 71, 73, 98, 106, 132, 139, 142, 152, 158, 
        166, 168, 174, 181, 192, 203, 205, 206, 207, 210, 225, 
        232, 251, 260, 269, 274, 281, 290, 293, 296, 305, 308, 
        388, 396, 401, 405, 444, 447, 509, 625, 663, 667, 676, 
        697, 792, 796, 980, 1031, 1034, 1040, 1124, 1142, 1146, 
        1157, 1207, 1209, 1301, 1316, 1344, 1485, 1512, 1583, 
        1616, 1623, 1635, 1714, 1736, 1739, 1744, 1758, 1816, 
        1855, 1864, 1871, 1872, 2121, 2128, 2260, 2301, 2305, 
        2308, 2344, 2345, 2354, 2368, 2545, 2745, 2749, 2756, 
        2769, 2771, 2772, 2773, 2774, 2788, 2799, 2809, 2811,
        2812, 2821, 2822, 2831, 2916, 2917, 2920, 2935, 2938, 
        2941, 2965, 2998, 3012, 3013, 3016, 3017, 3019
    ],
}

bad2good_idx = bad2good_idxs[ecg_time_str]
new_labels = labels.copy()
new_labels[bad2good_idx] = 2
new_labels = new_labels.astype(int)

print(
    f"Good: {len(new_labels[new_labels == 0])},", 
    f"Bad: {len(new_labels[new_labels == 1])},",
    f"Bad2good: {len(new_labels[new_labels == 2])}",
    )

label_map = {0: "good", 1: "bad", 2: "bad2good"}
str_labels = [label_map[label] for label in new_labels]

Good: 2849, Bad: 54, Bad2good: 118


## 6. 移动图片

In [36]:
import shutil, os

base_path = f"./ecg_figs/{ecg_time_str}/"
bad_path = base_path + "bad/"
bad2good_path = base_path + "bad2good/"
os.makedirs(bad2good_path, exist_ok=True)

for idx in bad2good_idx:
    idx_s = str(idx)
    bad_file = bad_path + idx_s + '_chunk_bad.jpg'
    bad2good_file = bad2good_path + idx_s + '_chunk_bad.jpg'
    shutil.move(bad_file, bad2good_file)

## 7. 保存标注 parquet

In [37]:
plabel_dict = {idx: [chunks[idx]['ecg'].values, str_labels[idx]] for idx in range(len(chunks))}
plabel_df = pd.DataFrame(plabel_dict).T
plabel_df.columns = ['ecg_raw', 'label']

plabel_df.to_parquet(f"{data_path}labeled_chunks_{ecg_time_str}.parquet")